In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast
from matplotlib.ticker import MaxNLocator

# ----------------------------------------------
# Configuration – ADAPT TO YOUR PATHS AND FORMAT
# ----------------------------------------------
base_path = '../data/agnews/400_steps/evolution/'   # where your CSV files live
seeds = [42, 123, 999, 2024, 7, 2026, 22]           # all seeds you have
run_ids = [1, 2, 3, 4, 5, 6, 7]                     # corresponding run ids
STEPS = 400                                          # used in CSV names
prefix = 'results_run_'                              # file prefix
suffix = f'_{STEPS}_steps.csv'                       # file suffix

OUTPUT_DIR = base_path + 'results/'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------
# 1. Load all runs and parse genotypes
# ----------------------------------------------
def parse_genotype(gen_str):
    try:
        return ast.literal_eval(gen_str)
    except:
        return []

all_runs = []          # list of DataFrames, one per run
for rid, seed in zip(run_ids, seeds):
    fname = f'{base_path}{prefix}{rid}_seed_{seed}{suffix}'
    df = pd.read_csv(fname)
    df['genotype_list'] = df['genotype'].apply(parse_genotype)
    df['num_0'] = df['genotype_list'].apply(lambda g: g.count(0))
    df['num_1'] = df['genotype_list'].apply(lambda g: g.count(1))
    df['depth'] = df['genotype_list'].apply(len)
    all_runs.append(df)

# ----------------------------------------------
# 2. Aggregate within each run (per generation)
# ----------------------------------------------
def aggregate_run(df):
    gen = df.groupby('generation')
    agg = gen.agg(
        fitness_mean = ('fitness', 'mean'),
        fitness_max  = ('fitness', 'max'),
        f1_mean      = ('f1', 'mean'),
        f1_max       = ('f1', 'max'),
        depth_mean   = ('depth', 'mean'),
        depth_max    = ('depth', 'max'),
        num_0_mean   = ('num_0', 'mean'),
        num_1_mean   = ('num_1', 'mean'),
        latency_mean = ('latency', 'mean')
    ).reset_index()
    agg['pct_0'] = agg['num_0_mean'] / (agg['num_0_mean'] + agg['num_1_mean']) * 100
    agg['pct_1'] = 100 - agg['pct_0']
    return agg

agg_per_run = [aggregate_run(df) for df in all_runs]

# Top-10% statistics per generation (within each run)
def top10_agg(df):
    # get indices of top 10% fitness per generation
    top10_list = []
    for gen, group in df.groupby('generation'):
        top = group.nlargest(max(1, int(0.1 * len(group))), 'fitness')
        top10_list.append(top)
    top10_df = pd.concat(top10_list)
    gen_top = top10_df.groupby('generation').agg(
        num_0_mean = ('num_0', 'mean'),
        num_1_mean = ('num_1', 'mean'),
        depth_mean = ('depth', 'mean')
    ).reset_index()
    gen_top['pct_0'] = gen_top['num_0_mean'] / (gen_top['num_0_mean'] + gen_top['num_1_mean']) * 100
    gen_top['pct_1'] = 100 - gen_top['pct_0']
    return gen_top

top10_per_run = [top10_agg(df) for df in all_runs]

# ----------------------------------------------
# 3. Aggregate across runs (mean & std per generation)
# ----------------------------------------------
def across_runs(metrics_list, columns):
    # metrics_list: list of DataFrames (one per run) with same columns
    # returns DataFrame with gen, mean, std for each column
    merged = pd.concat(metrics_list)
    grouped = merged.groupby('generation')
    result = pd.DataFrame()
    for col in columns:
        result[col + '_mean'] = grouped[col].mean()
        result[col + '_std']  = grouped[col].std()
    result = result.reset_index()
    return result

pop_columns = ['fitness_mean', 'fitness_max', 'f1_mean', 'f1_max',
               'depth_mean', 'depth_max', 'pct_0', 'pct_1', 'latency_mean']
pop_across = across_runs(agg_per_run, pop_columns)

top_columns = ['pct_0', 'pct_1', 'depth_mean']
top_across = across_runs(top10_per_run, top_columns)

# ----------------------------------------------
# 4. Plotting
# ----------------------------------------------
plt.style.use('seaborn-v0_8-darkgrid')

# Helper function
def plot_with_band(ax, x, y_mean, y_std, label, color, alpha=0.3):
    ax.plot(x, y_mean, label=label, color=color)
    ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=alpha, color=color)


# ---------- Figure 1: Fitness & F1 ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
plot_with_band(ax, pop_across['generation'], 
               pop_across['fitness_mean_mean'], pop_across['fitness_mean_std'],
               'Mean Fitness', 'blue')
plot_with_band(ax, pop_across['generation'], 
               pop_across['fitness_max_mean'], pop_across['fitness_max_std'],
               'Max Fitness', 'red')
ax.set_title('Fitness Evolution (across runs)')
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness')
ax.legend()

ax = axes[1]
plot_with_band(ax, pop_across['generation'], 
               pop_across['f1_mean_mean'], pop_across['f1_mean_std'],
               'Mean F1', 'green')
plot_with_band(ax, pop_across['generation'], 
               pop_across['f1_max_mean'], pop_across['f1_max_std'],
               'Max F1', 'orange')
ax.set_title('F1 Evolution (across runs)')
ax.set_xlabel('Generation')
ax.set_ylabel('Weighted F1')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'fitness_f1_evolution.png', dpi=150)
plt.close()

# ---------- Figure 2: Depth evolution ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
plot_with_band(ax, pop_across['generation'], 
               pop_across['depth_mean_mean'], pop_across['depth_mean_std'],
               'Mean Depth (pop)', 'purple')
ax.set_title('Average Depth (Population)')
ax.set_xlabel('Generation')
ax.set_ylabel('Layers')
ax.legend()

ax = axes[1]
plot_with_band(ax, pop_across['generation'], 
               pop_across['depth_max_mean'], pop_across['depth_max_std'],
               'Max Depth (pop)', 'brown')
ax.set_title('Max Depth (Population)')
ax.set_xlabel('Generation')
ax.set_ylabel('Layers')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'depth_evolution.png', dpi=150)
plt.close()

# ---------- Figure 3: Mamba/Attention % (whole population) ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.fill_between(pop_across['generation'], 0, pop_across['pct_0_mean'], 
                label='Mamba (0)', color='skyblue', alpha=0.6)
ax.fill_between(pop_across['generation'], pop_across['pct_0_mean'], 100,
                label='Attention (1)', color='salmon', alpha=0.6)
ax.set_title('Layer Type Proportion (Population)')
ax.set_xlabel('Generation')
ax.set_ylabel('Percentage (%)')
ax.legend()

ax = axes[1]
plot_with_band(ax, pop_across['generation'], 
               pop_across['pct_0_mean'], pop_across['pct_0_std'],
               'Mamba %', 'blue')
plot_with_band(ax, pop_across['generation'], 
               pop_across['pct_1_mean'], pop_across['pct_1_std'],
               'Attention %', 'red')
ax.set_title('Layer Type Proportion (with std)')
ax.set_xlabel('Generation')
ax.set_ylabel('Percentage (%)')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'proportion_population.png', dpi=150)
plt.close()

# ---------- Figure 4: Mamba/Attention % (top 10% best) ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.fill_between(top_across['generation'], 0, top_across['pct_0_mean'],
                label='Mamba (0)', color='skyblue', alpha=0.6)
ax.fill_between(top_across['generation'], top_across['pct_0_mean'], 100,
                label='Attention (1)', color='salmon', alpha=0.6)
ax.set_title('Layer Type Proportion (Top 10%)')
ax.set_xlabel('Generation')
ax.set_ylabel('Percentage (%)')
ax.legend()

ax = axes[1]
plot_with_band(ax, top_across['generation'], 
               top_across['pct_0_mean'], top_across['pct_0_std'],
               'Mamba %', 'blue')
plot_with_band(ax, top_across['generation'], 
               top_across['pct_1_mean'], top_across['pct_1_std'],
               'Attention %', 'red')
ax.set_title('Layer Type Proportion (Top 10%) with std')
ax.set_xlabel('Generation')
ax.set_ylabel('Percentage (%)')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'proportion_top10.png', dpi=150)
plt.close()

# ---------- Figure 5: Latency evolution ----------
fig, ax = plt.subplots(figsize=(10, 5))
plot_with_band(ax, pop_across['generation'], 
               pop_across['latency_mean_mean'], pop_across['latency_mean_std'],
               'Mean Latency', 'gray')
ax.set_title('Average Latency Evolution (across runs)')
ax.set_xlabel('Generation')
ax.set_ylabel('Latency (s/sample)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'latency_evolution.png', dpi=150)
plt.close()

# ---------- Figure 6: Final generation depth vs F1 (all individuals) ----------
# ... (unchanged)

print(f"All plots saved in {OUTPUT_DIR}")

All plots saved in ../data/agnews/400_steps/evolution/results/
